# Module 16 — Data over HTTP

The same sensor log as module 08, arriving over a socket instead of off a disk. Which
changes one thing and it is the thing this module is about: **on disk you decide the
encoding, over HTTP somebody else claims it** — and when they do not, `requests`
guesses, and guesses badly.

No network is involved. `server.py` next to this notebook answers the requests, and
it is worth reading: nine lines of logic, and the other side of everything below.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "server.py").is_file() else Path.cwd() / "16_http"
sys.path.append(str(HERE))  # so that `import server` finds it -- module 10

import requests  # noqa: E402
from server import serve  # noqa: E402

with serve() as base:
    print("the server is at", base)

## 1. HTTP in one cell

A request is a **method**, a **URL** and some **headers**. A response is a **status
code**, some **headers**, and a body that is **bytes**. That is the whole protocol as
far as this module is concerned.

In [ ]:
with serve() as base:
    response = requests.get(f"{base}/readings.csv", timeout=5)

print(response.status_code, response.reason)
print(response.headers["Content-Type"])
print(type(response.content).__name__, len(response.content), "bytes")
print(type(response.text).__name__, len(response.text), "code points")

Two bytes more than code points — the two `°` in the file, which is module 07's
arithmetic for the third time and the last.

The status codes worth knowing by their class rather than their number:

| | |
| --- | --- |
| **2xx** | it worked. 200 OK, 201 Created, 204 No Content |
| **3xx** | it is somewhere else. `requests` follows these by default |
| **4xx** | **you** got it wrong. 404 not found, 401 not authenticated, 403 not allowed, 429 too many requests |
| **5xx** | **the server** got it wrong. 500, 502, 503 |

The 4xx/5xx split is the one to keep: it says whose problem it is, and therefore
whether retrying could possibly help.

## 2. Three ways to read the body, and they are not interchangeable

`.content` is the bytes that arrived. `.text` is those bytes decoded. `.json()` is
`.text` parsed. Each step can go wrong on its own.

In [ ]:
with serve() as base:
    response = requests.get(f"{base}/readings.csv", timeout=5)

print(repr(response.content[:24]))  # bytes -- what came over the wire
print(repr(response.text[:24]))  # str -- decoded with response.encoding
print(response.content.decode("utf-8")[:24] == response.text[:24])  # the same operation

In [ ]:
with serve() as base:
    response = requests.get(f"{base}/readings.json", timeout=5)

payload = response.json()  # module 08's json.loads, on response.text
print(type(payload).__name__, sorted(payload))
print(payload["readings"][1])
print(type(payload["readings"][1]["value"]).__name__)  # a real float, not a string

Note the difference from module 08's CSV: a CSV gives you strings and you convert.
JSON carries types, so `91.0` arrives as a `float` — which is one reason an API
usually speaks JSON and a file usually does not.

`.json()` raises when the body is not JSON, and the exception is worth recognising
because the cause is almost always "the server sent an error page and I did not look
at the status code first".

In [ ]:
with serve() as base:
    response = requests.get(f"{base}/readings.csv", timeout=5)  # CSV, not JSON

try:
    response.json()
    outcome = "parsed"
except Exception as err:
    outcome = type(err).__name__

# What does .json() raise on a body that is not JSON?
assert outcome == ...

## 3. Where the encoding comes from

This is the section. On disk, module 08 wrote `encoding="utf-8"` and that was the end
of it. Over HTTP the encoding is **claimed in a header** — and the claim is optional.

The server here serves the same bytes on two paths. One says `charset=utf-8`, the
other says only `text/csv`. Predict both.

In [ ]:
with serve() as base:
    labelled = requests.get(f"{base}/readings.csv", timeout=5)
    unlabelled = requests.get(f"{base}/unlabelled.csv", timeout=5)

print(labelled.headers["Content-Type"])
print(unlabelled.headers["Content-Type"])

# Same bytes. What does each one say its encoding is?
assert labelled.encoding == ...
assert unlabelled.encoding == ...

In [ ]:
with serve() as base:
    unlabelled = requests.get(f"{base}/unlabelled.csv", timeout=5)

print(repr(unlabelled.text.splitlines()[1]))  # and this is what you get

`Â°C`. Module 08's silent failure, arriving from a different direction: no exception,
an ordinary `str`, and the damage travels on into your database.

The cause is a rule from an obsolete specification that `requests` still follows: for
a `text/*` response with no `charset`, the encoding is **ISO-8859-1** — which, as
module 08 established, can decode any byte sequence and therefore never fails.

Three ways to deal with it, in order of how much you should like them:

1. **Say it yourself** when you know: `response.encoding = "utf-8"` before touching
   `.text`. Correct, explicit, one line.
2. **Ask `requests` to look at the bytes**: `response.encoding =
   response.apparent_encoding` runs a character-set detector. Better than the
   fallback, still a guess.
3. **Decode the bytes yourself**: `response.content.decode("utf-8")`, which skips
   `.text` entirely and is the honest version of (1).

In [ ]:
with serve() as base:
    response = requests.get(f"{base}/unlabelled.csv", timeout=5)

    print("as sent:      ", repr(response.text.splitlines()[1]))

    response.encoding = "utf-8"  # set it before reading .text
    print("told the truth:", repr(response.text.splitlines()[1]))

    print("detected:     ", response.apparent_encoding)
    print("from bytes:   ", repr(response.content.decode("utf-8").splitlines()[1]))

And the rule to take away: **`.content` is what arrived; everything else is an
interpretation.** When it matters, decode the bytes yourself and name the encoding —
exactly as module 08 said, for exactly the same reason.

## 4. A 404 is not an exception

`requests.get` returns a response for a 404 and for a 500. It raises only when the
request could not be made at all. That trips people up, because "not found" feels
like an error and is not.

In [ ]:
with serve() as base:
    missing = requests.get(f"{base}/missing", timeout=5)

print(missing.status_code)

# Did the get() raise? And what does .ok say?
assert missing.ok == ...
assert missing.status_code == ...

In [ ]:
with serve() as base:
    missing = requests.get(f"{base}/missing", timeout=5)

    try:
        missing.raise_for_status()  # this is the line that turns 4xx and 5xx into exceptions
    except requests.HTTPError as err:
        print(type(err).__name__)
        print(str(err).split(" for url")[0])

So the shape of a request you would actually ship:

```python
response = requests.get(url, timeout=10)
response.raise_for_status()          # 4xx and 5xx become HTTPError
data = response.json()
```

Three lines, and the middle one is the one everybody forgets. Without it, a 500 that
returns an HTML error page reaches `.json()`, and the traceback you get is about JSON
rather than about the server being down.

## 5. Two kinds of failure

The distinction that matters when something goes wrong at three in the morning:

- **The server answered, and the answer is bad.** A status code came back. That is an
  `HTTPError`, and only if you asked for one.
- **There was no answer.** No route to the host, connection refused, DNS failure,
  nothing within the timeout. Those raise by themselves, and they are subclasses of
  `requests.RequestException`.

In [ ]:
try:
    requests.get("http://127.0.0.1:1/nothing-here", timeout=0.5)
except requests.ConnectionError as err:
    print("ConnectionError -- nobody answered")
    print(isinstance(err, requests.RequestException))

`requests.RequestException` is the base class, so `except requests.RequestException`
catches both kinds. Whether you want that depends on whether the two need different
handling — a connection error may be worth retrying, a 404 almost never is.

## 6. `timeout=` is not optional

Leave it out and there is **no** timeout: the call waits as long as the other side
keeps the socket open, which can be forever. A program that hangs with no traceback
and no log line is worse to debug than one that crashes, and this is the most common
way to write one.

In [ ]:
with serve() as base:
    try:
        requests.get(f"{base}/slow", timeout=0.2)  # the route sleeps for a second
    except requests.Timeout as err:
        print(type(err).__name__, "-- gave up after 0.2 s")

    slow = requests.get(f"{base}/slow", timeout=5)  # patient enough
    print(slow.status_code, repr(slow.text))

`ruff` will not tell you and `mypy` will not tell you: `timeout` is an ordinary
optional argument whose default means "wait forever". Every `requests` call in this
module has one, and so should every one you write.

## 7. Query parameters and headers

Do not build a URL by concatenating strings. `params=` takes a dict and encodes it —
which matters the moment a value contains a space, an ampersand or a non-ASCII
character.

In [ ]:
prepared = requests.Request(
    "GET",
    "http://example.invalid/readings",
    params={"tag": "TH 04", "unit": "°C", "limit": 85},
).prepare()

print(prepared.url)  # the spaces and the ° are encoded for you
print(prepared.method)

Read what it produced. The space became `+` — the form-encoding convention for a
query string, not `%20` — and `°` became `%C2%B0`, which is the two UTF-8 bytes of
that character written out, from module 07. An `&` in a value becomes `%26` so it
cannot be mistaken for the separator, and a literal `+` becomes `%2B` so it cannot be
mistaken for a space.

Every one of those is a rule you would have to know to build the URL by hand, and
getting one wrong produces a bug that appears for one customer whose name has an
umlaut in it.

Headers work the same way, as a dict:

```python
requests.get(url, headers={"Accept": "application/json"}, timeout=10)
```

And when you make more than one request to the same host, a `Session` reuses the
connection and carries the headers:

```python
with requests.Session() as session:
    session.headers["Accept"] = "application/json"
    for tag in tags:
        session.get(url, params={"tag": tag}, timeout=10)
```

That is a context manager, so module 09 covers why it is written that way.

## 8. The whole chain

`bytes` on the wire → `str` → `dict`, which is modules 07, 08 and this one in one
expression. Every arrow is a decode or a parse, and every arrow can fail.

In [ ]:
with serve() as base:
    response = requests.get(f"{base}/readings.json", timeout=10)
    response.raise_for_status()

    print("wire :", type(response.content).__name__)
    print("text :", type(response.text).__name__)
    print("data :", type(response.json()).__name__)

    faults = [r["tag"] for r in response.json()["readings"] if r["value"] > 85]
    print("faults:", faults)

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run. They all use `server.py`, so none of them needs
a network.

Module 17 is the other way to get data off the web: when there is no API and the
numbers are in a table on a page.